# Fine-Tuning de Produção: Moirai-MoE Bayesiano para Criptomoedas

**Objetivo:** Este notebook implementa um pipeline de fine-tuning robusto e pronto para produção para o modelo Moirai-MoE com um head Bayesiano, focado na previsão de séries temporais de criptomoedas.

**Estrutura de Produção:**
1.  **Configuração e Setup Inicial:**
    - Importação de bibliotecas.
    - Configuração de logging.
    - Definição de constantes e paths.
    - Fixação de `seeds` para reprodutibilidade.

2.  **Validação do Ambiente:**
    - Verificação de versões de Python, CUDA e bibliotecas essenciais.

3.  **Carregamento e Validação de Dados:**
    - Download de dados de alta frequência (1 minuto) da Binance.
    - Validação rigorosa dos dados: schema, tipos (`Float64`), e integridade.

4.  **Pré-processamento e Criação de Datasets:**
    - Utilização do `CryptoDatasetBuilder` com configurações SOTA (Estado da Arte).
    - Geração de datasets de treino, validação e teste.

5.  **Configuração do Modelo e Treinamento:**
    - Carregamento e ajuste da configuração a partir de um arquivo `YAML`.
    - Documentação clara dos hiperparâmetros.

6.  **Pipeline de Treinamento com PyTorch Lightning:**
    - Uso de callbacks essenciais: `ModelCheckpoint`, `EarlyStopping`, `LearningRateMonitor`.
    - Execução do treinamento monitorado.

7.  **Avaliação de Performance e Análise de Incerteza:**
    - Carregamento do melhor modelo a partir do checkpoint.
    - Cálculo de métricas de performance (CRPS, MAE, RMSE).
    - Visualização das previsões e dos intervalos de incerteza.

8.  **Salvamento do Artefato do Modelo:**
    - Exportação do modelo treinado para deployment.

### Requisitos de Ambiente

**Atenção:** Este notebook assume que o ambiente já está configurado. Se você estiver executando pela primeira vez, garanta que todas as dependências estão instaladas.

1.  **Clone o repositório (se ainda não o fez):**
    ```bash
    git clone https://github.com/waldefran/uni2ts.git
    cd uni2ts
    ```

2.  **Instale as dependências:**
    ```bash
    pip install -e .
    pip install -r requirements_crypto.txt
    ```

In [ ]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3.11.5 ('uni2ts')
#     language: python
#     name: python3
# ---

# # 1. Configuração e Setup Inicial
# ## 1.1. Importação das Bibliotecas

import os
import sys
import yaml
import logging
import torch
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from pathlib import Path

# Adiciona o diretório 'src' ao path para importações locais
# NOTA: Ajuste o path se a estrutura do seu projeto for diferente
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

print(f"Versão do PyTorch: {torch.__version__}")
print(f"Versão do Lightning: {pl.__version__}")

In [ ]:
# ## 1.2. Configuração de Logging Estruturado

# Configura um logger para registrar informações de forma clara e padronizada,
# essencial para debugging em ambientes de produção.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

logger.info("Logging configurado com sucesso.")

In [ ]:
# ## 1.3. Definição de Constantes e Paths

# Centralizar paths e constantes melhora a manutenibilidade e evita hardcoding.
# Usamos a biblioteca `pathlib` para uma manipulação de paths mais robusta.

# Path base do projeto (assumindo que o notebook está na raiz)
# Se o notebook estiver em outro lugar, ajuste o `Path.cwd()`
PROJECT_ROOT = Path.cwd()
logger.info(f"Raiz do projeto definida em: {PROJECT_ROOT}")

# Paths para dados, logs e saídas
DATA_DIR = PROJECT_ROOT / "data"
BINANCE_DATA_DIR = DATA_DIR / "binance_data"
OUTPUT_DIR = PROJECT_ROOT / "output"
LOG_DIR = PROJECT_ROOT / "logs"
CONFIG_PATH = PROJECT_ROOT / "configs" / "crypto" / "finetune_bayesian_moe.yaml"

# Cria os diretórios se eles não existirem
DATA_DIR.mkdir(exist_ok=True)
BINANCE_DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# Constantes do Modelo
# COMENTÁRIO: Estes são os ativos que vamos usar. Para um teste rápido,
# você pode reduzir a lista, por exemplo, para `["BTCUSDT", "ETHUSDT"]`.
ASSETS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT"]
# Tipo de dado para precisão numérica. Float64 é recomendado para dados financeiros.
DTYPE = "float64"

logger.info(f"Diretórios de trabalho criados/verificados.")
logger.info(f"Ativos para o estudo: {ASSETS}")
logger.info(f"Tipo de dado a ser usado: {DTYPE}")

In [ ]:
# ## 1.4. Fixação de Seeds para Reprodutibilidade

# Fixar as seeds é crucial para garantir que os resultados do treinamento
# e da avaliação sejam os mesmos a cada execução.
def set_seed(seed):
    """Fixa as seeds para PyTorch, NumPy e Python."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    pl.seed_everything(seed)
    logger.info(f"Seeds fixadas em {seed} para garantir reprodutibilidade.")

SEED = 42
set_seed(SEED)

# 2. Validação do Ambiente

Esta seção garante que o ambiente de execução possui os componentes necessários. Em um pipeline de produção, esta etapa pode ser automatizada para evitar falhas inesperadas.

In [ ]:
# ## 2.1. Verificação de Hardware e Software

logger.info("Iniciando validação do ambiente...")

# Verificar versão do Python
python_version = sys.version.split()[0]
logger.info(f"Versão do Python: {python_version}")
if not "3.11" in python_version:
    logger.warning("A versão 3.11 do Python é a recomendada para compatibilidade total.")

# Verificar disponibilidade de GPU
is_cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count()
logger.info(f"CUDA disponível: {is_cuda_available}")
if is_cuda_available:
    logger.info(f"Número de GPUs: {gpu_count}")
    for i in range(gpu_count):
        logger.info(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    logger.warning("Nenhuma GPU detectada. O treinamento será executado em CPU, o que pode ser muito lento.")

# Verificar se o TF32 está habilitado em GPUs Ampere ou mais recentes
# COMENTÁRIO: TF32 oferece um bom balanço entre performance e precisão.
# Pode ser desabilitado para máxima precisão, se necessário.
if is_cuda_available and torch.cuda.get_device_capability()[0] >= 8:
    torch.set_float32_matmul_precision('high') # 'high' ou 'medium'
    logger.info("TF32 (TensorFloat-32) habilitado para aceleração em GPUs compatíveis.")

logger.info("Validação do ambiente concluída.")

# 3. Carregamento e Validação de Dados

A qualidade dos dados é fundamental. Aqui, baixamos os dados e realizamos uma validação rigorosa para garantir que eles atendem aos requisitos do modelo.

In [ ]:
# ## 3.1. Download dos Dados da Binance (Intervalo: 1 minuto)

from binanceDataloader import BinanceDataDownloader

logger.info("Iniciando download dos dados de 1 minuto da Binance...")
downloader = BinanceDataDownloader(output_dir=str(BINANCE_DATA_DIR))

# COMENTÁRIO: `years_back=0.5` baixa os últimos 6 meses de dados.
# Para um treinamento mais robusto, considere aumentar este valor (ex: 1.0 para 1 ano).
# Para testes rápidos, você pode diminuir para 0.1 (aprox. 1 mês).
results = downloader.download_all_symbols(
    symbols=ASSETS,
    years_back=0.5
)

logger.info("Download concluído. Iniciando validação dos arquivos.")

# ## 3.2. Validação dos Arquivos Baixados
parquet_files = list(BINANCE_DATA_DIR.rglob("*.parquet"))
if not parquet_files:
    raise RuntimeError(f"Nenhum arquivo .parquet encontrado em {BINANCE_DATA_DIR}. Verifique o processo de download.")

logger.info(f"Encontrados {len(parquet_files)} arquivos .parquet para validação.")

required_columns = ['open_time', 'open', 'high', 'low', 'close', 'volume']
validated_files = 0

for file_path in parquet_files:
    try:
        df = pd.read_parquet(file_path)
        asset_name = file_path.parent.name
        
        # 1. Validação de Schema
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            logger.error(f"Arquivo {file_path.name} para o ativo {asset_name} tem colunas faltando: {missing_cols}")
            continue
            
        # 2. Validação de Dados Nulos
        if df[required_columns].isnull().values.any():
            logger.warning(f"Arquivo {file_path.name} para o ativo {asset_name} contém valores nulos. Considere um tratamento.")
            # Ação recomendada: df.interpolate(method='linear', inplace=True) ou df.dropna(inplace=True)
            
        # 3. Validação de Tipos (garantindo Float64)
        for col in ['open', 'high', 'low', 'close', 'volume']:
            if df[col].dtype != 'float64':
                df[col] = df[col].astype('float64')
        
        # 4. Validação de `open_time`
        if not pd.api.types.is_datetime64_any_dtype(df['open_time']):
             df['open_time'] = pd.to_datetime(df['open_time'])

        logger.info(f"Arquivo {file_path.name} ({asset_name}) validado com sucesso. {len(df):,} linhas.")
        validated_files += 1
        
    except Exception as e:
        logger.error(f"Falha ao validar o arquivo {file_path}: {e}")

if validated_files < len(ASSETS):
    raise ValueError("Nem todos os ativos foram validados com sucesso. Verifique os logs de erro.")

logger.info("Validação de dados concluída com sucesso.")

# 4. Pré-processamento e Criação de Datasets

Com os dados validados, usamos o `CryptoDatasetBuilder` para transformá-los em um formato que o modelo Moirai possa consumir. Esta etapa aplica features de estado da arte.

In [ ]:
# ## 4.1. Configuração do CryptoDatasetBuilder

from uni2ts.data.builder.crypto import CryptoDatasetBuilder, CryptoConfig

# COMENTÁRIO: Estes parâmetros são cruciais para a performance do modelo.
# - `context_length`: Quantos passos no tempo o modelo "vê" para fazer uma previsão. (24h de dados de 1min)
# - `prediction_length`: Quantos passos no tempo o modelo prevê. (1h de previsão)
# - `unified_dataset`: Trata todos os ativos como um único grande dataset, melhorando a generalização.
# - `window_normalization`: Normaliza os dados por janela, focando na forma da série temporal.
crypto_config = CryptoConfig(
    context_length=1440,
    prediction_length=60,
    unified_dataset=True,
    anonymous_training=True,
    window_normalization=True,
    cyclical_features=True,
    min_sequence_length=1440 + 60,
    validation_split=0.2,
    dtype=DTYPE,
    target_assets=ASSETS,
)

logger.info("Configuração do CryptoDatasetBuilder definida.")
logger.info(f"Contexto: {crypto_config.context_length} min, Previsão: {crypto_config.prediction_length} min")

# ## 4.2. Construção dos Datasets
logger.info("Iniciando a construção dos datasets...")

dataset_builder = CryptoDatasetBuilder(
    data_path=str(BINANCE_DATA_DIR),
    config=crypto_config
)

# Valida se a configuração é compatível com os dados encontrados
if not dataset_builder.validate_config(check_data_path=True):
    raise ValueError("A configuração do dataset builder é inválida para os dados fornecidos.")

train_dataset, val_dataset, test_dataset = dataset_builder.build_datasets()

logger.info("Datasets construídos com sucesso.")
logger.info(f"Tamanho do dataset de treino: {len(train_dataset)}")
logger.info(f"Tamanho do dataset de validação: {len(val_dataset)}")
logger.info(f"Tamanho do dataset de teste: {len(test_dataset)}")

# Validar uma amostra
sample = train_dataset[0]
logger.info(f"Formato da amostra de dados: {list(sample.keys())}")
logger.info(f"Shape do target: {sample['target'].shape}")
logger.info(f"Tipo de dado do target: {sample['target'].dtype}")

# 5. Configuração do Modelo e Treinamento

Carregamos a configuração base de um arquivo YAML e a ajustamos para nosso pipeline de produção. Manter a configuração em arquivos facilita a experimentação e o versionamento.

In [ ]:
# ## 5.1. Carregamento e Ajuste da Configuração YAML

logger.info(f"Carregando configuração do modelo de: {CONFIG_PATH}")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# --- AJUSTES DE PRODUÇÃO ---
# COMENTÁRIO: Aqui, sobrescrevemos a configuração do arquivo YAML com os
# parâmetros definidos neste notebook, garantindo consistência.

# Paths e dados
config["data"]["data_path"] = str(BINANCE_DATA_DIR)
config["data"]["config"]["target_assets"] = ASSETS
config["data"]["config"]["dtype"] = DTYPE

# Dataloaders
# COMENTÁRIO: `batch_size` depende da memória da sua GPU. Se encontrar erros de "Out of Memory",
# reduza este valor. `num_workers` acelera o carregamento dos dados.
config["train_dataloader"]["batch_size"] = 16
config["train_dataloader"]["num_workers"] = os.cpu_count() // 2
config["val_dataloader"]["batch_size"] = 16
config["val_dataloader"]["num_workers"] = os.cpu_count() // 2

# Trainer
# COMENTÁRIO: `max_epochs` define o número máximo de épocas de treinamento.
# O callback `EarlyStopping` pode parar o treinamento antes se o modelo parar de melhorar.
config["trainer"]["max_epochs"] = 20
config["trainer"]["default_root_dir"] = str(LOG_DIR)

# Modelo
config["model"]["context_length"] = crypto_config.context_length
config["model"]["prediction_length"] = crypto_config.prediction_length

# Salvar a configuração final usada para este treinamento
final_config_path = OUTPUT_DIR / "final_production_config.yaml"
with open(final_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

logger.info(f"Configuração final salva em: {final_config_path}")

# 6. Pipeline de Treinamento com PyTorch Lightning

Usamos o `pytorch_lightning` para um loop de treinamento limpo e poderoso. Os `callbacks` automatizam tarefas como salvar checkpoints, parar o treinamento mais cedo e monitorar taxas de aprendizado.

In [ ]:
from uni2ts.cli.train import train

# ## 6.1. Configuração dos Callbacks de Produção

# COMENTÁRIO: Estes callbacks são essenciais para um treinamento robusto.
# - `ModelCheckpoint`: Salva o melhor modelo com base em uma métrica (ex: `val_loss`).
# - `EarlyStopping`: Evita overfitting parando o treinamento quando a performance para de melhorar.
# - `LearningRateMonitor`: Loga a taxa de aprendizado, útil para debugging.
# - `BayesianUncertaintyMonitor`: Callback customizado para visualizar a incerteza do modelo.

config["callbacks"].append({
    "_target_": "pytorch_lightning.callbacks.ModelCheckpoint",
    "monitor": "val_loss",
    "mode": "min",
    "save_top_k": 1,
    "dirpath": str(OUTPUT_DIR / "checkpoints"),
    "filename": "best-model-{epoch:02d}-{val_loss:.4f}",
})

config["callbacks"].append({
    "_target_": "pytorch_lightning.callbacks.EarlyStopping",
    "monitor": "val_loss",
    "patience": 5, # COMENTÁRIO: Número de épocas sem melhora antes de parar.
    "mode": "min",
})

config["callbacks"].append({
    "_target_": "pytorch_lightning.callbacks.LearningRateMonitor",
    "logging_interval": "step",
})

# Ajusta o path do plot_dir no callback de incerteza
for callback_conf in config["callbacks"]:
    if "BayesianUncertaintyMonitor" in callback_conf.get("_target_", ""):
        callback_conf["plot_dir"] = str(OUTPUT_DIR / "uncertainty_plots")
        (OUTPUT_DIR / "uncertainty_plots").mkdir(exist_ok=True)
        logger.info("Path para plots de incerteza configurado.")

# ## 6.2. Execução do Treinamento
logger.info("Iniciando o pipeline de treinamento...")

# A função `train` do uni2ts encapsula a lógica do PyTorch Lightning
# e usa a configuração que preparamos.
train(config)

logger.info("Treinamento concluído.")

# 7. Avaliação de Performance e Análise de Incerteza

Após o treinamento, avaliamos o modelo no conjunto de teste para obter uma estimativa imparcial de sua performance. Também visualizamos as previsões para uma análise qualitativa.

In [ ]:
from uni2ts.cli.eval import evaluate
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

logger.info("Iniciando a avaliação do modelo no conjunto de teste...")

# ## 7.1. Carregamento do Melhor Modelo
# Encontra o melhor checkpoint salvo pelo callback
checkpoint_dir = OUTPUT_DIR / "checkpoints"
best_model_path = next(checkpoint_dir.glob("best-model-*.ckpt"), None)

if not best_model_path:
    raise FileNotFoundError("Nenhum checkpoint do modelo foi encontrado. Verifique se o treinamento foi concluído com sucesso.")

logger.info(f"Carregando o melhor modelo de: {best_model_path}")

# ## 7.2. Execução da Avaliação
# A função `evaluate` carrega o modelo e o executa no dataloader de teste.
# COMENTÁRIO: `num_samples` controla o número de amostras Monte Carlo para estimar a incerteza.
# Um valor maior (ex: 100) dá uma estimativa mais estável, mas é mais lento.
eval_results = evaluate(
    weight_path=str(best_model_path),
    config=config,
    num_samples=100,
)

logger.info("Avaliação concluída.")

# ## 7.3. Análise das Métricas de Performance
# O CRPS (Continuous Ranked Probability Score) é uma métrica chave para previsões probabilísticas.
logger.info("Métricas de Performance no Conjunto de Teste:")
for key, value in eval_results["metrics"].items():
    logger.info(f"  - {key}: {value:.6f}")

# ## 7.4. Visualização das Previsões e Incerteza
logger.info("Gerando visualizações das previsões...")

# Extrai os dados para plotagem
forecasts = eval_results["forecasts"]
targets = eval_results["targets"]
item_ids = eval_results["item_ids"]

# COMENTÁRIO: Vamos visualizar as primeiras 4 previsões.
# Você pode mudar o `num_plots` ou selecionar `item_ids` específicos.
num_plots = 4
fig, axes = plt.subplots(num_plots, 1, figsize=(15, 5 * num_plots), sharex=True)
fig.suptitle("Previsões do Modelo vs. Valores Reais (Conjunto de Teste)", fontsize=16)

for i in range(num_plots):
    ax = axes[i]
    item_id = item_ids[i]
    target = targets[i]
    forecast = forecasts[i]
    
    # Calcula quantis para o intervalo de confiança
    q5 = np.quantile(forecast, 0.05, axis=0)
    q95 = np.quantile(forecast, 0.95, axis=0)
    median = np.quantile(forecast, 0.5, axis=0)
    
    prediction_len = len(target)
    time_steps = np.arange(prediction_len)
    
    ax.plot(time_steps, target, label="Valor Real", color="black")
    ax.plot(time_steps, median, label="Previsão (Mediana)", color="blue", linestyle="--")
    ax.fill_between(time_steps, q5, q95, color="blue", alpha=0.2, label="Intervalo de Confiança (90%)")
    
    ax.set_title(f"Ativo: {item_id}")
    ax.set_ylabel("Preço Normalizado")
    ax.legend()

axes[-1].set_xlabel("Passos de Tempo Futuros")
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plot_path = OUTPUT_DIR / "prediction_plots.png"
plt.savefig(plot_path)
logger.info(f"Gráficos de previsão salvos em: {plot_path}")
plt.show()

# 8. Salvamento do Artefato do Modelo

Finalmente, salvamos o modelo treinado em um formato padrão (como ONNX ou TorchScript) para que possa ser facilmente carregado em um ambiente de produção para inferência em tempo real.

**Nota:** A exportação para ONNX/TorchScript pode exigir ajustes no código do modelo se ele contiver operações não padrão. Esta seção fornece um exemplo básico.

In [ ]:
from uni2ts.model.moirai import MoiraiForecast

logger.info("Iniciando exportação do modelo para produção...")

# Carrega o módulo Lightning a partir do checkpoint
lightning_module = MoiraiForecast.load_from_checkpoint(best_model_path)
model = lightning_module.model
model.eval() # Coloca o modelo em modo de avaliação

# Cria um exemplo de entrada (dummy input) com o shape esperado
batch_size = 1
context_len = config["model"]["context_length"]
# O número de features dinâmicas depende do `CryptoDatasetBuilder`
# Vamos pegar de uma amostra do dataset
num_features = test_dataset[0]['feat_dynamic_real'].shape[0]

dummy_input = {
    "past_target": torch.randn(batch_size, context_len),
    "past_observed_target": torch.ones(batch_size, context_len, dtype=torch.bool),
    "feat_dynamic_real": torch.randn(batch_size, context_len + config["model"]["prediction_length"], num_features)
}

# Converte todos os inputs para o tipo de dado correto (Float64)
for key, tensor in dummy_input.items():
    if torch.is_floating_point(tensor):
        dummy_input[key] = tensor.to(getattr(torch, DTYPE))


# Exporta para TorchScript
try:
    scripted_model = torch.jit.script(model)
    torchscript_path = OUTPUT_DIR / "moirai_production_model.pt"
    scripted_model.save(str(torchscript_path))
    logger.info(f"Modelo exportado com sucesso para TorchScript em: {torchscript_path}")
except Exception as e:
    logger.error(f"Falha ao exportar para TorchScript: {e}")
    logger.warning("A exportação para TorchScript pode falhar com modelos complexos. O checkpoint (.ckpt) ainda é o artefato principal.")

# O arquivo .ckpt salvo pelo PyTorch Lightning já é um artefato de produção robusto
# e pode ser carregado diretamente para inferência, como fizemos na etapa de avaliação.
logger.info("Processo de fine-tuning de produção concluído.")